In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from loaders._load_vn30_reg import preprocess, VN30, TARGETS
from sklearn.linear_model import MultiTaskLassoCV, LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_percentage_error

In [6]:
_ = preprocess("ACB", verbose=True)

=== Preprocessing ACB ===
Feature shapes in train: (1215, 120), val: (0, 120), test: (328, 120)
Target shapes in train: (1215, 4), val: (0, 4), test: (328, 4)


In [7]:
track = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=2)
    X_train, Y_train = data['train']
    X_val, Y_val = data['val']
    X_test, Y_test = data['test']
    target_scaler = data['scaler']['target']

    tscv = TimeSeriesSplit(n_splits=3)
    model = MultiTaskLassoCV(cv=tscv)
    model.fit(X_train, Y_train)

    Y_pred = model.predict(X_test)
    Y_pred = target_scaler.inverse_transform(Y_pred)
    Y_test = target_scaler.inverse_transform(Y_test)

    r2 = r2_score(Y_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_test, Y_pred) * 100

    track["r2"].append(r2)
    track["mape"].append(mape)

    print(f"Symbol: {symbol}, R^2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R^2: {np.mean(track['r2']):.4f}, Mean MAPE: {np.mean(track['mape']):.4f}")
print(f"Std R^2: {np.std(track['r2']):.4f}, Std MAPE: {np.std(track['mape']):.4f}")

Symbol: ACB, R^2: 0.9569, MAPE: 0.7122
Symbol: BCM, R^2: 0.9648, MAPE: 1.1670
Symbol: BID, R^2: 0.9350, MAPE: 0.8833
Symbol: BVH, R^2: 0.9839, MAPE: 0.9220
Symbol: CTG, R^2: 0.9764, MAPE: 0.8877
Symbol: FPT, R^2: 0.9920, MAPE: 1.0348
Symbol: GAS, R^2: 0.9623, MAPE: 0.7380
Symbol: GVR, R^2: 0.9780, MAPE: 1.3500
Symbol: HDB, R^2: 0.9823, MAPE: 0.9064
Symbol: HPG, R^2: 0.9260, MAPE: 0.8854
Symbol: LPB, R^2: 0.9969, MAPE: 1.0408
Symbol: MBB, R^2: 0.9684, MAPE: 0.8804
Symbol: MSN, R^2: 0.9665, MAPE: 0.9711
Symbol: MWG, R^2: 0.9871, MAPE: 1.0242
Symbol: PLX, R^2: 0.9855, MAPE: 0.9157
Symbol: SAB, R^2: 0.9448, MAPE: 0.8584
Symbol: SHB, R^2: 0.9680, MAPE: 0.9753
Symbol: SSB, R^2: 0.9678, MAPE: 0.8053
Symbol: SSI, R^2: 0.9472, MAPE: 1.0531
Symbol: STB, R^2: 0.9840, MAPE: 0.9588
Symbol: TCB, R^2: 0.9847, MAPE: 0.9270
Symbol: TPB, R^2: 0.9622, MAPE: 0.9973
Symbol: VCB, R^2: 0.9044, MAPE: 0.6784
Symbol: VHM, R^2: 0.9754, MAPE: 1.0026
Symbol: VIB, R^2: 0.9446, MAPE: 0.9019
Symbol: VIC, R^2: 0.9780,

In [9]:
track = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=2)
    X_train, Y_train = data['train']
    X_val, Y_val = data['val']
    X_test, Y_test = data['test']
    target_scaler = data['scaler']['target']

    tscv = TimeSeriesSplit(n_splits=3)
    
    n_tasks = Y_train.shape[1]
    Y_pred = np.zeros_like(Y_test)
    for i in range(n_tasks):
        model = LassoCV(cv=tscv)
        Y_train_i = Y_train[:, i]
        model.fit(X_train, Y_train_i)
        Y_pred_i = model.predict(X_test)
        Y_pred[:, i] = Y_pred_i

    Y_pred = target_scaler.inverse_transform(Y_pred)
    Y_test = target_scaler.inverse_transform(Y_test)

    r2 = r2_score(Y_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_test, Y_pred) * 100

    track["r2"].append(r2)
    track["mape"].append(mape)

    print(f"Symbol: {symbol}, R^2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R^2: {np.mean(track['r2']):.4f}, Mean MAPE: {np.mean(track['mape']):.4f}")
print(f"Std R^2: {np.std(track['r2']):.4f}, Std MAPE: {np.std(track['mape']):.4f}")

Symbol: ACB, R^2: 0.9574, MAPE: 0.6922
Symbol: BCM, R^2: 0.9653, MAPE: 1.1309
Symbol: BID, R^2: 0.9355, MAPE: 0.8728
Symbol: BVH, R^2: 0.9841, MAPE: 0.8965
Symbol: CTG, R^2: 0.9764, MAPE: 0.8820
Symbol: FPT, R^2: 0.9920, MAPE: 1.0248
Symbol: GAS, R^2: 0.9629, MAPE: 0.7229
Symbol: GVR, R^2: 0.9782, MAPE: 1.3396
Symbol: HDB, R^2: 0.9825, MAPE: 0.8906
Symbol: HPG, R^2: 0.9282, MAPE: 0.8562
Symbol: LPB, R^2: 0.9969, MAPE: 1.0193
Symbol: MBB, R^2: 0.9687, MAPE: 0.8653
Symbol: MSN, R^2: 0.9668, MAPE: 0.9540
Symbol: MWG, R^2: 0.9872, MAPE: 1.0115
Symbol: PLX, R^2: 0.9856, MAPE: 0.9055
Symbol: SAB, R^2: 0.9416, MAPE: 0.8716
Symbol: SHB, R^2: 0.9694, MAPE: 0.9225
Symbol: SSB, R^2: 0.9674, MAPE: 0.8033
Symbol: SSI, R^2: 0.9502, MAPE: 0.9933
Symbol: STB, R^2: 0.9845, MAPE: 0.9268
Symbol: TCB, R^2: 0.9848, MAPE: 0.9152
Symbol: TPB, R^2: 0.9633, MAPE: 0.9585
Symbol: VCB, R^2: 0.9070, MAPE: 0.6553
Symbol: VHM, R^2: 0.9759, MAPE: 0.9627
Symbol: VIB, R^2: 0.9484, MAPE: 0.8388
Symbol: VIC, R^2: 0.9794,